In [46]:
# 下载输入文件（莎士比亚作品全文）
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [47]:
# 打开输入文件
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
# 查看输入文件的长度
print(f'length of dataset in character is {len(text)}')

length of dataset in character is 1115394


In [48]:
# 先统计出现了多少种字符
vocabulary = sorted(list(set(text)))
print(f'the size of vacabulary is {len(vocabulary)}')
print(f'what is in vocabulary : {vocabulary}')

the size of vacabulary is 65
what is in vocabulary : ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [49]:
# Tokenizer，将输入映射成为一张词汇表，由编码器和解码器组成

## 1. 构建String To Integer 的映射字典
stoi = { ch : i for i, ch in enumerate(vocabulary) }
## 2. 构建Integer To String 的映射字段
itos = { i : ch for i, ch in enumerate(vocabulary) }

## 3. Encoder 将输入的字符串转化为Token List
def encoder(x):
    return [stoi[ch] for ch in x]

## 4. Decoder 将Token List 转化为人能读的字符串
def decoder(digits_list):
    return ''.join([itos[i] for i in digits_list])

In [50]:
encoded_text = encoder('Hello, World!')
print(f'Encoder {'Hello, World!'} to integer list is {encoded_text}')
print(f'Deocde {encoded_text} to text is {decoder(encoded_text)}')

Encoder Hello, World! to integer list is [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Deocde [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2] to text is Hello, World!


In [55]:
# 将输入文件加载为张量
import torch

data = torch.tensor(encoder(text), dtype=torch.long)
print(data)

print(f'tensor shape of data : {data.shape}')
print(f'tensor type of data : {data.dtype}')

tensor([18, 47, 56,  ..., 45,  8,  0])
tensor shape of data : torch.Size([1115394])
tensor type of data : torch.int64


In [56]:
# 划分训练集和测试集
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

In [61]:
# 构造一个批次的输入
context_length = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i + context_length] for i in ix])
    y = torch.stack([data[i + 1: i + context_length + 1] for i in ix])

    return x, y

xb, yb = get_batch('train')
print(f'x:{xb}')
print(xb.dtype, xb.shape)
print(f'y:{yb}')
print(yb.dtype, yb.shape)

x:tensor([[58, 46, 43, 43,  6,  1, 47, 58],
        [ 8,  0, 15, 39, 50, 50,  1, 59],
        [14, 53, 63,  6,  1, 58, 46, 47],
        [ 1, 40, 43,  1, 46, 43, 56,  1]])
torch.int64 torch.Size([4, 8])
y:tensor([[46, 43, 43,  6,  1, 47, 58,  1],
        [ 0, 15, 39, 50, 50,  1, 59, 54],
        [53, 63,  6,  1, 58, 46, 47, 57],
        [40, 43,  1, 46, 43, 56,  1, 50]])
torch.int64 torch.Size([4, 8])


In [65]:
# 构建BigramLanguageModel
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_tale = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_tale(idx)

        if targets == None:
            return logits
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

            return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits = self(idx)

            logits = logits[:, -1, :]
            prob = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(prob, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

BGLM = BigramLanguageModel(len(vocabulary))

logits, loss = BGLM(xb, yb)
print(logits.shape, loss)

print(decoder(BGLM.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65]) tensor(4.5297, grad_fn=<NllLossBackward0>)

tOjR.L,O.F:z3yQesxnaxT,MuFlPXyUk SFlFnvZ;Ic,IdmqJBgVnv-;mp,hG
Of'ENeMr?sWdap-dRyvCj!n,ZU: K !yd !nJb
